In [0]:
import os, sys, re
import pandas as pd

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, repo_root)

from src import config_loader as cl
from src import meta

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "", "Catalog override (optional)")
dbutils.widgets.text("run_id", "", "Run id (blank = generate)")
dbutils.widgets.text("pipeline_name", "manual_gold", "Pipeline name")

g = lambda k: dbutils.widgets.get(k).strip()
run_id = g("run_id") or cl.new_batch_id()
pipeline_name = g("pipeline_name")
overrides = {"project": {"catalog": g("catalog")}} if g("catalog") else None
cfg = cl.load_config(os.path.join(repo_root, "config", "config.yaml"), overrides)
print("run_id:", run_id)

In [0]:
meta.create_meta_tables(spark, cfg)

catalog = cfg["project"]["catalog"]
layers = cfg["storage"]["layers"]
GOLD = f"{catalog}.{layers['gold']}"
tokens = {"{silver}": f"{catalog}.{layers['silver']}", "{gold}": GOLD,
          "{date_start}": str(cfg["gold"]["date_dim_start"]),
          "{date_end}": str(cfg["gold"]["date_dim_end"])}

sql_text = open(os.path.join(repo_root, "sql", "gold_views.sql")).read()
for k, v in tokens.items():
    sql_text = sql_text.replace(k, v)
statements = [s.strip() for s in sql_text.split(";") if "CREATE OR REPLACE VIEW" in s.upper()]

results = []
for stmt in statements:
    m = re.search(r"CREATE OR REPLACE VIEW\s+\S+\.(\w+)", stmt, re.I)
    name = m.group(1) if m else "unknown"
    t0 = cl.utc_now()
    status, error, n = "SUCCESS", None, None
    try:
        spark.sql(stmt)
        n = spark.table(f"{GOLD}.{name}").count()
    except Exception as e:
        status, error = "FAILED", f"{type(e).__name__}: {e}"
    t1 = cl.utc_now()
    meta.log_run(spark, cfg, run_id=run_id, pipeline_name=pipeline_name, stage="GOLD",
                 resource_type=name, load_date=None, status=status, started_at=t0, ended_at=t1,
                 data_saved_at=t1 if status == "SUCCESS" else None,
                 records_written=n, error_message=error)
    results.append({"view": name, "status": status, "rows": n, "error": error})

res_df = pd.DataFrame(results)
display(res_df)
if (res_df["status"] != "SUCCESS").any():
    raise Exception(f"Gold view problems:\n{res_df[res_df['status'] != 'SUCCESS'].to_string()}")

In [0]:
pairs = [("Patient", "dim_patient"), ("Encounter", "fact_encounter"),
         ("Observation", "fact_observation"), ("Condition", "fact_condition")]
rows = []
for resource, view in pairs:
    s = spark.sql(f"SELECT count(*) FROM {cl.resource_table(cfg, 'silver', resource)} WHERE is_current").collect()[0][0]
    gcount = spark.table(f"{GOLD}.{view}").count()
    rows.append({"resource": resource, "silver_current_rows": s, "gold_view": view,
                 "gold_rows": gcount, "match": s == gcount})
display(pd.DataFrame(rows))

In [0]:
display(spark.table(f"{GOLD}.vw_encounters_by_month").orderBy("year_month"))
display(spark.table(f"{GOLD}.vw_top_conditions").orderBy("condition_count", ascending=False).limit(10))
display(spark.table(f"{GOLD}.dim_patient").limit(10))